# EJERCICIO SQL: CERVEZAS
## 1. Tablas de la Base de Datos

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
os.getcwd()

'c:\\Users\\urkow\\Documents\\Documentos_Clase\\2026-02-BILBAO-FT-Data-Science\\2-Data_Analysis\\8-BBDD\\SQL\\Teoria\\SQL_Python\\ejercicio'

In [2]:
# Conectamos con la base de datos chinook.db
connection = sqlite3.connect('data/cervezas.db')

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [4]:
# Borramos tabla para pruebas
query = f'drop table cervezas'
crsr.execute(query);
query = f'drop table bares'
crsr.execute(query);
query = f'drop table empleados'
crsr.execute(query);
query = f'drop table reparto'
crsr.execute(query);

### Creamos tablas e insertamos registros

In [5]:
tablas = {
    'cervezas': {
        'columnas': ('CodC', 'Envase', 'Capacidad', 'Stock'),
        'tipos':    ('NVARCHAR(2) PRIMARY KEY NOT NULL', 'NVARCHAR(25)', 'DECIMAL(5, 2)', 'INTEGER'),
        'datos': [{'CodC':'01','Envase':'Botella','Capacidad':0.2,'Stock':3600},
                  {'CodC':'02','Envase':'Botella','Capacidad':0.33,'Stock':1200},
                  {'CodC':'03','Envase':'Lata','Capacidad':0.33,'Stock':2400},
                  {'CodC':'04','Envase':'Botella','Capacidad':1,'Stock':288},
                  {'CodC':'05','Envase':'Barril','Capacidad':60,'Stock':3}
        ]
    },
    'bares': {
        'columnas': ('CodB', 'Cif', 'Nombre', 'Localidad'),
        'tipos':    ('NVARCHAR(3) PRIMARY KEY NOT NULL', 'NVARCHAR(9)', 'NVARCHAR(40) NOT NULL', 'NVARCHAR(20)'),
        'datos': [{'CodB':'001','Cif':'11111111X','Nombre':'Stop','Localidad':'Villa Botijo'},
                  {'CodB':'002','Cif':'22222222Y','Nombre':'Las Vegas','Localidad':'Villa Botijo'},
                  {'CodB':'003','Cif':np.nan,'Nombre':'Club Social','Localidad':'Las Ranas'},
                  {'CodB':'004','Cif':'33333333Z','Nombre':'Otra Ronda','Localidad':'La Esponja'}
        ]
    },
    'empleados': {
        'columnas': ('CodE', 'Nombre', 'Sueldo'),
        'tipos':    ('INTEGER PRIMARY KEY NOT NULL', 'NVARCHAR(25) NOT NULL', 'INTEGER'),
        'datos': [{'CodE':1,'Nombre':'Prudencio Caminero','Sueldo':120000},
                  {'CodE':2,'Nombre':'Vicente Merario','Sueldo':110000},
                  {'CodE':3,'Nombre':'Valentin Siempre','Sueldo':100000}
        ]
    },
    'reparto': {
        'columnas': ('CodE', 'CodB', 'CodC', 'Fecha', 'Cantidad'),
        'tipos':    ('INTEGER NOT NULL', 'NVARCHAR(3) NOT NULL', 'NVARCHAR(3) NOT NULL', 'DATETIME NOT NULL', 'INTEGER NOT NULL'),
        'foreign_keys': [('CodE', 'empleados', 'CodE', ''),
                         ('CodC',   'cervezas', 'CodC',   ''),
                         ('CodB',   'bares', 'CodB',   '')
        ],
        'datos': [{'CodE':1,'CodB':'001','CodC':'01','Fecha':'2005/10/21','Cantidad':240},
                  {'CodE':1,'CodB':'001','CodC':'02','Fecha':'2005/10/21','Cantidad':48},
                  {'CodE':1,'CodB':'002','CodC':'03','Fecha':'2005/10/22','Cantidad':60},
                  {'CodE':1,'CodB':'004','CodC':'05','Fecha':'2005/10/22','Cantidad':4},
                  {'CodE':2,'CodB':'002','CodC':'03','Fecha':'2005/10/22','Cantidad':48},
                  {'CodE':2,'CodB':'002','CodC':'05','Fecha':'2005/10/23','Cantidad':2},
                  {'CodE':2,'CodB':'004','CodC':'01','Fecha':'2005/10/23','Cantidad':480},
                  {'CodE':2,'CodB':'004','CodC':'02','Fecha':'2005/10/24','Cantidad':72},
                  {'CodE':3,'CodB':'003','CodC':'03','Fecha':'2005/10/24','Cantidad':48},
                  {'CodE':3,'CodB':'003','CodC':'04','Fecha':'2005/10/25','Cantidad':20}
        ]
    }
}

In [ ]:
for nom_tabla, info in tablas.items():
    columnas = info['columnas']
    tipos = info['tipos']
    datos = info['datos']

    # --- Crear tabla dinámicamente ---
    definiciones = ', '.join(f'{col} {tipo}' for col, tipo in zip(columnas, tipos))
    sql_cre = f'CREATE TABLE IF NOT EXISTS {nom_tabla} ({definiciones})'
    crsr.execute(sql_cre)

    # --- Query insert ---
    cols_sql = ', '.join(columnas)
    values = ', '.join(['?'] * len(columnas))
    sql_ins = f'INSERT INTO {nom_tabla} ({cols_sql}) VALUES ({values})'

    # --- Insertar registros ---
    for fila in datos:
        valores = tuple(fila[col] for col in columnas)
        crsr.execute(sql_ins, valores)

connection.commit()

**Explicación de zip**

In [21]:
lista1 = ('A','B','C')
lista2 = (0, 1, 2)

zip_ejemplo = ', '.join(f'{i} {j}' for i, j in zip(lista1, lista2))

zip_ejemplo

'A 0, B 1, C 2'

**Explicación de    ', '.join(['?'] * len(columnas))**

values = ", ".join(["?"] * len(columnas))  
4 columnas --> values = '?, ?, ?, ?'  
  
sql_ins = INSERT INTO cervezas VALUES (?, ?, ?, ?)  
  
si tenemos valores = ('01', 'Botella', 0.2, 3600)  
  
**crsr.execute(sql_ins, valores)** --> INSERT INTO cervezas VALUES ('01', 'Botella', 0.2, 3600)

In [12]:
# Comprobamos que se ha creado bien la tabla
pd.read_sql_query('SELECT * FROM reparto;',connection)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,2005/10/21,240
1,1,001,02,2005/10/21,48
2,1,002,03,2005/10/22,60
3,1,004,05,2005/10/22,4
4,2,002,03,2005/10/22,48
5,2,002,05,2005/10/23,2
6,2,004,01,2005/10/23,480
7,2,004,02,2005/10/24,72
8,3,003,03,2005/10/24,48
9,3,003,04,2005/10/25,20


In [1]:
connection.close()

NameError: name 'connection' is not defined

## 2. Enunciados de los Ejercicios

In [4]:
# Conectamos con la base de datos chinook.db
connection = sqlite3.connect('data/cervezas.db')

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()

In [5]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.


In [22]:
query = '''
select distinct emp.nombre as [Nombre empleado]
    from empleados as emp
        left join reparto as rep on rep.code = emp.code
        right join bares as bar on bar.codb = rep.codb
    where bar.nombre = 'Stop'
      and rep.fecha between '2005/10/17' and '2005/10/23'
'''
sql_query(query)

,Nombre empleado
0,Prudencio Caminero


2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.

In [42]:
query = '''
select distinct bar.cif as Cif, bar.nombre as Bar
    from reparto as rep
        right join cervezas as cer on rep.codc = cer.codc
        inner join bares as bar on bar.codb = rep.codb
    where cer.envase = 'Botella'
      and cer.capacidad < 1
    order by bar.localidad
'''
sql_query(query)

,Cif,Bar
0,33333333Z,Otra Ronda
1,11111111X,Stop


3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.


In [25]:
query = '''
select bar.nombre as Bar, concat(cer.envase, ' (', cer.capacidad, ')') as Envase, rep.fecha as Fecha, rep.cantidad as Cantidad
    from reparto as rep
        right join empleados as emp on emp.code = rep.code
        inner join bares as bar on bar.codb = rep.codb
        inner join cervezas as cer on cer.codc = rep.codc
    where emp.nombre = 'Prudencio Caminero';
'''
sql_query(query)

,Bar,Envase,Fecha,Cantidad
0,Stop,Botella (0.2),2005/10/21,240
1,Stop,Botella (0.33),2005/10/21,48
2,Las Vegas,Lata (0.33),2005/10/22,60
3,Otra Ronda,Barril (60),2005/10/22,4


4. Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.


In [26]:
query = '''
select bar.nombre as Bar, concat(cer.envase, ' (', cer.capacidad, ')') as Envase
    from reparto as rep
        right join empleados as emp on emp.code = rep.code
        inner join bares as bar on bar.codb = rep.codb
        inner join cervezas as cer on cer.codc = rep.codc
    where cer.envase = 'Botella'
      and cer.capacidad in (0.2,0.33);
'''
sql_query(query)

,Bar,Envase
0,Stop,Botella (0.2)
1,Stop,Botella (0.33)
2,Otra Ronda,Botella (0.2)
3,Otra Ronda,Botella (0.33)


5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella. 

In [28]:
query = '''
select emp.nombre as Empleado
    from reparto as rep
        right join empleados as emp on emp.code = rep.code
        inner join bares as bar on bar.codb = rep.codb
        inner join cervezas as cer on cer.codc = rep.codc
    where bar.nombre in ('Stop','Las Vegas')
      and cer.envase = 'Botella'
    group by Empleado;
'''
sql_query(query)

,Empleado
0,Prudencio Caminero


6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.


In [ ]:
query = '''
select emp.nombre as Empleado, count(rep.fecha) as Viajes
    from reparto as rep
        right join empleados as emp on emp.code = rep.code
        inner join bares as bar on bar.codb = rep.codb
    where bar.localidad <> 'Villa Botijo'
    group by Empleado;
'''
sql_query(query)

,Empleado,Viajes
0,Prudencio Caminero,1
1,Valentin Siempre,2
2,Vicente Merario,2


7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.


In [41]:
query = '''
select bar.nombre as Bar, bar.localidad, sum(rep.cantidad * cer.capacidad) as [Litros de cerveza]
    from reparto as rep
        right join empleados as emp on emp.code = rep.code
        inner join bares as bar on bar.codb = rep.codb
        inner join cervezas as cer on cer.codc = rep.codc
    group by Bar
    order by [Litros de cerveza] desc
    limit 1;
'''
sql_query(query)

,Bar,Localidad,Litros de cerveza
0,Otra Ronda,La Esponja,359.76


8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.


In [44]:
query = '''
select distinct bar.nombre as Bar
    from reparto as rep
        right join cervezas as cer on rep.codc = cer.codc
        inner join bares as bar on bar.codb = rep.codb
    where cer.envase = 'Botella'
      and cer.capacidad < 1
'''
sql_query(query)

,Bar
0,Stop
1,Otra Ronda


9. Subir un 5% el sueldo del empleado que más días haya trabajado.


Resetear sueldos:  
```sql
query = '''
update empleados
set sueldo = 120000
where code = 1;
'''
crsr.execute(query)

query = '''
update empleados
set sueldo = 110000
where code = 2;
'''
crsr.execute(query)

query = '''
update empleados
set sueldo = 100000
where code = 3;
'''
crsr.execute(query);
```

In [18]:
# Miramos tabla
query = '''
select *
    from empleados;
'''
sql_query(query)

,CodE,Nombre,Sueldo
0,1,Prudencio Caminero,126000
1,2,Vicente Merario,110000
2,3,Valentin Siempre,100000


In [ ]:
# Entiendo trabajar más como mayor número de viajes, hay 2 empleados!!!
query = '''
update empleados
set sueldo = 1.05 * sueldo
where nombre in (
    select emp.nombre as Empleado
        from reparto as rep
            right join empleados as emp on emp.code = rep.code
        group by emp.nombre        
        having count(rep.fecha) = (
            select max(contador)
                from (
                    select count(rep.fecha) as contador
                        from reparto as rep
                        group by rep.code
                )
        )
);
'''
crsr.execute(query)


In [ ]:
# Comprobamos update
query = '''
select *
    from empleados;
'''
sql_query(query)


,CodE,Nombre,Sueldo
0,1,Prudencio Caminero,126000
1,2,Vicente Merario,110000
2,3,Valentin Siempre,100000
3,4,Vicente Merario,90000


In [ ]:
connection.commit() # Guardar los cambios

Si consideramos más litros repartidos la query sería así:

```sql
update empleados
set sueldo = 1.05 * sueldo
where nombre in (
    select emp.nombre as Empleado
        from reparto as rep
            right join empleados as emp on emp.code = rep.code
            inner join cervezas as cer on cer.codc = rep.codc
        group by emp.nombre        
        having sum(rep.cantidad * cer.capacidad) = (
            select max(contador)
                from (
                    select sum(rep.cantidad * cer.capacidad) as contador
                        from reparto as rep
                            inner join cervezas as cer on cer.codc = rep.codc
                        group by rep.code
                )
        )
);
```

10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.

In [ ]:
# Para que el empleado pueda repartir, lo insertamos en la tabla de EMPLEADOS
query = '''
insert into empleados
(CodE, Nombre, Sueldo) 
values (4, 'Vicente Merario', 90000)
'''
crsr.execute(query)

In [24]:
# Comprobamos insert
query = '''
select *
    from empleados;
'''
sql_query(query)


,CodE,Nombre,Sueldo
0,1,Prudencio Caminero,126000
1,2,Vicente Merario,110000
2,3,Valentin Siempre,100000
3,4,Vicente Merario,90000


In [ ]:
connection.commit() # Guardar los cambios

In [ ]:
# Registramos el pedido en la tabla de REPARTO
query = '''
insert into reparto
(CodE, CodB, CodC, Fecha, Cantidad)
values (4, '001', '03', '2005/10/26', 48)
'''
crsr.execute(query)

In [25]:
# Comprobamos insert
query = '''
select *
    from reparto;
'''
sql_query(query)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,2005/10/21,240
1,1,001,02,2005/10/21,48
2,1,002,03,2005/10/22,60
3,1,004,05,2005/10/22,4
4,2,002,03,2005/10/22,48
5,2,002,05,2005/10/23,2
6,2,004,01,2005/10/23,480
7,2,004,02,2005/10/24,72
8,3,003,03,2005/10/24,48
9,3,003,04,2005/10/25,20


In [26]:
connection.commit() # Guardar los cambios

In [27]:
connection.commit() # Commit por si nos queda algo pendiente antes de cerrar conexión
connection.close()